# 04 — Transfer learning

La pregunta del notebook es directa: dadas dos arquitecturas pre-entrenadas en ImageNet y dos estrategias de uso, ¿cuál combinación nos da el mejor clasificador? Probamos las cuatro variantes que tienen sentido:

|   | ResNet18 | EfficientNet-B0 |
|---|---|---|
| **Feature extraction** (backbone congelado, sólo se entrena la cabeza) | rápido, suelo bajo | rápido, suelo bajo |
| **Fine-tuning** (todos los pesos aprendibles, LR diferencial) | más caro pero techo alto | más caro, techo alto |

En la cabeza final usamos para todas las variantes la misma capa: `Linear(in_features → 256) → ReLU → Dropout(0.3) → Linear(256 → 196 clases)`. En el fine-tuning aplicamos un LR diferencial: 1e-5 para el backbone (cambios suaves sobre los pesos pre-entrenados) y 1e-3 para la cabeza (que parte de aleatorio y necesita aprender deprisa). Entrenamos cada configuración 10 épocas con batch 128 y mixed precision en GPU.

El modelo ganador se serializa en `saved_models/best_model.pt` junto con `best_model_config.json`, que es lo que cargará el notebook 06 para la demo interactiva.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch, numpy as np, random
torch.manual_seed(42); np.random.seed(42); random.seed(42)
torch.backends.cudnn.benchmark = True

METADATA_PATH = ROOT / 'data' / 'metadata.csv'
MODELS_DIR    = ROOT / 'saved_models'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

from src import (PretrainedModel, get_dataloaders, train_model,
                 plot_training_curves, full_evaluation_report,
                 plot_confusion_matrix, plot_per_class_f1, plot_roc_curves)

In [ ]:
EPOCHS = 10
BATCH = 128 if DEVICE == 'cuda' else 32
NUM_WORKERS = 4 if DEVICE == 'cuda' else 0

train_loader, val_loader, test_loader, class_names, class_to_idx = get_dataloaders(
    metadata_path=str(METADATA_PATH), batch_size=BATCH, num_workers=NUM_WORKERS, root_dir=ROOT,
)
NUM_CLASSES = len(class_names)
print(f'NUM_CLASSES = {NUM_CLASSES}')
print(f'Train batches: {len(train_loader)}  Val: {len(val_loader)}  Test: {len(test_loader)}')

In [ ]:
import pandas as pd

configs = [
    ('resnet18', 'feature_extraction'),
    ('resnet18', 'finetune'),
    ('efficientnet_b0', 'feature_extraction'),
    ('efficientnet_b0', 'finetune'),
]
results = {}
for backbone, strategy in configs:
    name = f'{backbone}_{strategy}'
    print(f'\n=== {name} ===')
    model = PretrainedModel(backbone=backbone, num_classes=NUM_CLASSES, strategy=strategy)
    if strategy == 'finetune':
        opt = torch.optim.Adam(model.get_optimizer_groups(lr_backbone=1e-5, lr_head=1e-3))
    else:
        opt = torch.optim.Adam(model.classifier.parameters(), lr=1e-3)
    save_path = str(MODELS_DIR / f'{name}.pt')
    h = train_model(model, train_loader, val_loader, epochs=EPOCHS, optimizer=opt,
                    device=DEVICE, patience=8, save_path=save_path, experiment_name=name)
    plot_training_curves(h, name)
    results[name] = (h, save_path, backbone, strategy)

In [ ]:
rows = []
for name, (h, sp, bk, st) in results.items():
    rows.append({'experiment': name, 'best_val_acc': max(h['val_acc']),
                 'best_epoch': h['best_epoch']})
comp = pd.DataFrame(rows).sort_values('best_val_acc', ascending=False)
display(comp)
best_name = comp.iloc[0]['experiment']
h_best, sp_best, bk_best, st_best = results[best_name]
print(f'\nGanador: {best_name}')

In [ ]:
model = PretrainedModel(backbone=bk_best, num_classes=NUM_CLASSES, strategy=st_best)
model.load_state_dict(torch.load(sp_best, map_location=DEVICE))
model = model.to(DEVICE)
rep = full_evaluation_report(model, test_loader, class_names, device=DEVICE)
print(f"Test accuracy: {rep['accuracy']:.3f}")
print(f"Macro F1:     {rep['macro_f1']:.3f}")
print(f"Weighted F1:  {rep['weighted_f1']:.3f}")
plot_confusion_matrix(rep['confusion_matrix'], class_names, title=f'Mejor modelo: {best_name}')
plot_per_class_f1(rep['report_df'])
plot_roc_curves(rep['y_true'], rep['y_prob'], class_names, max_classes=10)

## Sección "Image-to-Image" (aumentación procedural)

Esta sección genera variantes de tres compuestos pasándolos por el pipeline `AUGMENT_ONLY` de `Albumentations` (rotación, ruido gaussiano, deformación elástica, brillo/contraste). **Conviene aclarar que estas variantes son aumentación procedural, no generación con un modelo entrenado.** La parte realmente generativa del proyecto —entrenar un modelo capaz de producir imágenes nuevas— está en el notebook `04b_generative.ipynb` con un Conditional VAE. Esta sección queda aquí para demostrar visualmente qué tipo de variabilidad estamos inyectando en el train loader.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
from src.augmentation import AUGMENT_ONLY
df = pd.read_csv(METADATA_PATH)
show_ids = df['compound_id'].drop_duplicates().sample(3, random_state=0).tolist()

fig, axes = plt.subplots(3, 10, figsize=(18, 6))
for ax_row, cid in zip(axes, show_ids):
    fp = df[df['compound_id'] == cid].iloc[0]['filepath']
    base = np.array(Image.open(ROOT / fp).convert('RGB'))
    for j, ax in enumerate(ax_row):
        out = AUGMENT_ONLY(image=base)['image']
        ax.imshow(out); ax.axis('off')
        if j == 0: ax.set_ylabel(cid, fontsize=10)
    ax_row[0].set_title(cid, fontsize=10, loc='left')
plt.tight_layout(); plt.show()

In [ ]:
import json, shutil
best_path = MODELS_DIR / 'best_model.pt'
shutil.copy(sp_best, best_path)

config = {
    'architecture': 'PretrainedModel',
    'backbone': bk_best,
    'strategy': st_best,
    'num_classes': int(NUM_CLASSES),
    'class_names': list(class_names),
    'val_accuracy': float(max(h_best['val_acc'])),
    'test_accuracy': float(rep['accuracy']),
    'test_macro_f1': float(rep['macro_f1']),
}
with open(MODELS_DIR / 'best_model_config.json', 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
print(f'Modelo guardado en {best_path}')
print(json.dumps(config, indent=2, ensure_ascii=False)[:300])

## Análisis de los resultados

Las cuatro combinaciones nos dan estos `best val_acc` tras 10 épocas:

| Configuración | best val_acc | Ranking |
|---|---:|:---:|
| ResNet18 — feature extraction | 82,7% | 3º |
| **ResNet18 — fine-tune** | **99,4%** | **1º** ⭐ |
| EfficientNet-B0 — feature extraction | 81,5% | 4º |
| EfficientNet-B0 — fine-tune | 98,9% | 2º |

El mejor modelo (ResNet18 fine-tuned) llega a **99,4% accuracy y 99,4% F1 macro en test**, prácticamente perfecto.

Cuatro observaciones:

1. **Fine-tuning >> feature extraction, por 17 puntos.** Esta es la lección más nítida del experimento. Con feature extraction (backbone congelado), las dos arquitecturas se quedan en torno al 82%, ligeramente por encima del baseline ML clásico (69%) y muy lejos de la CNN propia del notebook 03 (98%). Esto nos dice que los embeddings de ImageNet capturan algo útil sobre formas geométricas, pero **no son específicos para estructuras químicas**. Hace falta dejar que los pesos del backbone se adapten al dominio para extraer la información discriminativa que de verdad nos sirve.

2. **ResNet18 fine-tuned bate a EfficientNet-B0 fine-tuned por medio punto.** En la teoría EfficientNet debería ser superior (mejor relación parámetros/accuracy en ImageNet), pero en este dataset relativamente pequeño y con sólo 10 épocas, la arquitectura más simple de ResNet18 converge más rápido y de forma más estable. EfficientNet probablemente cerraría el gap (o lo invertiría) con más épocas o un LR scheduler dedicado, pero para el coste/beneficio de este proyecto la elección es clara: **ResNet18 fine-tuned**.

3. **La CNN del notebook 03 ya estaba muy cerca.** Su Exp2 alcanzó 98,2% en validación con sólo 280k parámetros (frente a los 11M de ResNet18). Esto matiza un dogma habitual del DL: el transfer learning **no siempre** justifica su sobrecoste. Si tienes un dataset razonable, balanceado y con augmentación moderada, una CNN propia bien dimensionada puede rondar el mismo techo. Aquí ganamos 1,2 puntos (98,2% → 99,4%), valiosos cuando son los últimos antes del techo natural del problema, pero no espectaculares.

4. **Implicación para producción.** Como el modelo va a correr en la demo interactiva del notebook 06, lo que más importa es la latencia y el footprint. ResNet18 (45 MB en estado serializado) cabe cómodamente en memoria y hace inferencia en pocos ms en CPU, incluso sin GPU. EfficientNet-B0 es más liviano pero su latencia es comparable. Cualquiera de las dos opciones sería viable; con el tie-break del val_acc nos quedamos con ResNet18.

El modelo se guarda en `saved_models/best_model.pt` y su configuración (backbone, número de clases, accuracy de test, lista de `class_names`) en `best_model_config.json`. Ese par de ficheros es la única dependencia externa del notebook 06.